<a href="https://colab.research.google.com/github/ANKUM24/RuView/blob/main/emr%20iceberg%20tut1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

# Install PySpark 3.5.1 to be compatible with Iceberg 1.5.0
!pip install pyspark==3.5.1

# Set PYSPARK_SUBMIT_ARGS to include the Iceberg package for Spark 3.5
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0 pyspark-shell'


In [2]:
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [3]:

spark = SparkSession.builder \
    .appName("IcebergETL") \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "warehouse"
    ) \
    .config("spark.sql.warehouse.dir", "warehouse") \
    .enableHiveSupport() \
    .getOrCreate()


In [4]:
# Verify the installed Spark version
print(f"Spark Version: {spark.version}")

Spark Version: 3.5.1


In [5]:
# Read CSV
nyTaxi = spark.read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("tripdata.csv")


In [6]:

# Add column
updatedNYTaxi = nyTaxi.withColumn(
    "current_date",
    lit(datetime.now())
)

In [7]:

# Create Iceberg table
updatedNYTaxi.writeTo(
    "local.db.nytaxi"
).createOrReplace()

In [8]:
# Verify the installed Spark version
print(f"Spark Version: {spark.version}")

Spark Version: 3.5.1


In [9]:
# IMPORTANT: Run the following line in a SEPARATE cell, then RESTART YOUR COLAB RUNTIME (Runtime -> Restart runtime...) and run all cells.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2 pyspark-shell'

# Query back
spark.sql("SELECT * FROM local.db.nytaxi").show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|        current_date|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2|         1/1/17 0:01|          1/1/17 0:11|                 N|         1|          42|         166|              1|         1.71|        9.0|  0.0|    0.

In [11]:
# To display 50 rows:
print("Displaying the first 50 rows:")
spark.sql("SELECT * FROM local.db.nytaxi").show(n=50)

Displaying the first 50 rows:
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|        current_date|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2|         1/1/17 0:01|          1/1/17 0:11|                 N|         1|          42|         166|              1|        

In [13]:

# To display all rows (use with caution on very large datasets):
print(f"Displaying all {spark.sql("SELECT * FROM local.db.nytaxi").count()} rows:")
# spark.sql("SELECT * FROM local.db.nytaxi").show(numRows=spark.sql("SELECT * FROM local.db.nytaxi").count())


Displaying all 20000 rows:


In [14]:
df = spark.read.parquet("warehouse/db/nytaxi/data")
df.explain(True)

# The explain() output typically focuses on the execution plan (how Spark processes data),
# not the specific metadata of the underlying file format like compression codec.
# However, you can often infer the compression from the file names if they follow common conventions.

print("\n--- Checking compression for DataFrame 'df' ---")
# Check the input files to see if their names contain compression indicators
input_parquet_files = df.inputFiles()
if input_parquet_files:
    print("Input Parquet files:")
    for file_path in input_parquet_files:
        print(f"  - {file_path}")
        if ".snappy.parquet" in file_path.lower():
            print("    -> Snappy compression is indicated by the filename.")
        elif ".gzip.parquet" in file_path.lower():
            print("    -> Gzip compression is indicated by the filename.")
        elif ".lzo.parquet" in file_path.lower():
            print("    -> LZO compression is indicated by the filename.")
        else:
            # Spark writes Parquet files with Snappy compression by default if not specified
            # and may not always append '.snappy' if it's the default and not explicitly set.
            # In an Iceberg table context, the files are typically Snappy compressed by default.
            print("    -> No explicit compression suffix found in filename.")
            print("       Spark generally uses Snappy compression by default for Parquet files if not specified.")
else:
    print("No input files found for the DataFrame.")

# You can also check Spark's default Parquet compression configuration
print(f"\nSpark's default Parquet compression codec: {spark.conf.get('spark.sql.parquet.compression.codec', 'snappy (default)')}")


== Parsed Logical Plan ==
Relation [VendorID#546,lpep_pickup_datetime#547,lpep_dropoff_datetime#548,store_and_fwd_flag#549,RatecodeID#550,PULocationID#551,DOLocationID#552,passenger_count#553,trip_distance#554,fare_amount#555,extra#556,mta_tax#557,tip_amount#558,tolls_amount#559,ehail_fee#560,improvement_surcharge#561,total_amount#562,payment_type#563,trip_type#564,current_date#565] parquet

== Analyzed Logical Plan ==
VendorID: int, lpep_pickup_datetime: string, lpep_dropoff_datetime: string, store_and_fwd_flag: string, RatecodeID: int, PULocationID: int, DOLocationID: int, passenger_count: int, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: string, improvement_surcharge: double, total_amount: double, payment_type: int, trip_type: int, current_date: timestamp
Relation [VendorID#546,lpep_pickup_datetime#547,lpep_dropoff_datetime#548,store_and_fwd_flag#549,RatecodeID#550,PULocationID#551,DOLocationID#552,p

In [16]:

df1 = spark.read.parquet(
   "output_parquet"
)
df1.explain(True)

== Parsed Logical Plan ==
Relation [VendorID#586,lpep_pickup_datetime#587,lpep_dropoff_datetime#588,store_and_fwd_flag#589,RatecodeID#590,PULocationID#591,DOLocationID#592,passenger_count#593,trip_distance#594,fare_amount#595,extra#596,mta_tax#597,tip_amount#598,tolls_amount#599,ehail_fee#600,improvement_surcharge#601,total_amount#602,payment_type#603,trip_type#604,current_date#605] parquet

== Analyzed Logical Plan ==
VendorID: int, lpep_pickup_datetime: string, lpep_dropoff_datetime: string, store_and_fwd_flag: string, RatecodeID: int, PULocationID: int, DOLocationID: int, passenger_count: int, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: string, improvement_surcharge: double, total_amount: double, payment_type: int, trip_type: int, current_date: timestamp
Relation [VendorID#586,lpep_pickup_datetime#587,lpep_dropoff_datetime#588,store_and_fwd_flag#589,RatecodeID#590,PULocationID#591,DOLocationID#592,p

In [17]:
print(f"\nSpark's default Parquet compression codec: {spark.conf.get('spark.sql.parquet.compression.codec', 'snappy (default)')}")


IllegalArgumentException: The value of spark.sql.parquet.compression.codec should be one of brotli, uncompressed, lz4, gzip, lzo, lz4raw, snappy, lz4_raw, none, zstd, but was snappy (default)

In [ ]:
spark.read.format